# Machine Learning: Detailed Foundations and Reliable Practice

This notebook keeps the full conceptual detail of the original notes while organizing it into a single learning path. It combines mathematical intuition, vectorized implementations, visual diagnostics, and production-minded evaluation.

## Learning map

1. **Linear regression:** model, cost function, convexity, and gradient descent
2. **Multiple features:** vectorization, matrix notation, and the normal equation
3. **Optimization:** feature scaling, learning curves, and learning-rate diagnosis
4. **Feature engineering:** polynomial regression and model complexity
5. **Classification:** sigmoid, logistic loss, and decision boundaries
6. **Generalization:** bias, variance, regularization, leakage, and evaluation

## Shape convention

| Symbol | Meaning | Shape |
|---|---|---|
| $m$ | number of training examples | scalar |
| $n$ | number of features | scalar |
| $X$ | design matrix | `(m, n)` |
| $\mathbf{x}^{(i)}$ | one example | `(n,)` |
| $\mathbf{w}$ | weight vector | `(n,)` |
| $b$ | bias | scalar |
| $\hat{\mathbf y}$ | predictions | `(m,)` |

> When code fails, print every array's `.shape` before changing the mathematics.


In [1]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

rng = np.random.default_rng(42)
np.set_printoptions(precision=4, suppress=True)


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## Math bridge for this notebook

If partial derivatives, gradients, Jacobians, or the chain rule are unfamiliar, study [`math_foundations_for_ml.ipynb`](math_foundations_for_ml.ipynb) first. It explains these ideas in Chinese with geometric plots and step-by-step derivations.

You do not need advanced calculus before continuing. For each formula, keep three questions in view:

1. What are the inputs and output?
2. What does each derivative measure?
3. Does the gradient have the same shape as the parameter it updates?


# Part I — Linear Regression and Optimization

Linear regression is the cleanest place to understand the three components shared by most machine-learning algorithms:

1. a **model** that maps inputs to predictions;
2. an **objective function** that measures error;
3. an **optimization algorithm** that changes parameters to reduce that error.


## 1. The Cost Function $J(w,b)$ 

### 1. The Functional Purpose

The cost function is a **mathematical measure of performance**. While a loss function calculates error for one data point, $J(w,b)$ aggregates the error across the **entire training set** of $m$ examples.

* **Low $J$:** The model fits the data well.
* **High $J$:** The model's parameters $(w, b)$ are poorly chosen.

### 2. The Squared Error Logic

The standard formula for linear regression is the **Mean Squared Error (MSE)**:


$$J(w,b) = \frac{1}{2m} \sum_{i=1}^{m} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

* **Squaring:** Ensures all errors are positive and disproportionately penalizes larger outliers.
* **$\frac{1}{2m}$:** The $m$ provides an average; the $2$ is a constant that simplifies the math during differentiation (canceling the exponent).

### 3. The Convex Property

$J(w,b)$ is a **convex function** (bowl-shaped). This is logically critical because:

* It has **no local minima**, only one global minimum.
* It guarantees that optimization algorithms like **Gradient Descent** will always have a clear "downhill" path to the most accurate parameters.

---

**Key Insight:** $J(w,b)$ transforms the abstract goal of "learning" into a concrete task of **minimization**.


### Cost function vs. loss function

For one example, the squared loss is

$$
L^{(i)}=\frac12(\hat y^{(i)}-y^{(i)})^2.
$$

The cost function averages this loss over the dataset. Keeping the distinction clear is useful because training usually minimizes an aggregate objective, while error analysis often examines individual losses.


In [2]:
# Generate data from y = 2.5x - 1 + noise.
# X is kept two-dimensional: (samples, features).
X = rng.uniform(-3, 3, size=(120, 1))
y = 2.5 * X[:, 0] - 1.0 + rng.normal(0, 0.8, size=120)


def predict_linear(X: np.ndarray, w: np.ndarray, b: float) -> np.ndarray:
    """Return predictions with shape (m,) from X:(m,n) and w:(n,)."""
    return X @ w + b


def mse_cost(X: np.ndarray, y: np.ndarray,
             w: np.ndarray, b: float) -> float:
    """Return J(w,b) = mean squared error divided by 2."""
    errors = predict_linear(X, w, b) - y
    return float(np.mean(errors ** 2) / 2)


initial_w = np.zeros(X.shape[1])
print("X shape:", X.shape, "w shape:", initial_w.shape, "y shape:", y.shape)
print("Initial cost:", mse_cost(X, y, initial_w, b=0.0))


X shape: (120, 1) w shape: (1,) y shape: (120,)
Initial cost: 9.19917922136076


## 2. Gradient Descent: The Essentials

### 1. The Goal

Minimize the **Cost Function $J(w,b)$** (the error) by iteratively adjusting parameters $w$ and $b$.

### 2. The Update Rule

Go "downhill" by subtracting the gradient from the current value:


$$w = w - \alpha \frac{\partial}{\partial w} J(w,b)$$

$$b = b - \alpha \frac{\partial}{\partial b} J(w,b)$$

* **$\alpha$ (Learning Rate):** Your step size.
* **Small $\alpha$:** Reliable but slow.(baby step)
* **Large $\alpha$:** Fast, but may overshoot(exceeding a intended target, limit, or capacity) or never converge.



### 3. Simultaneous Update

**Crucial:** You must calculate the gradients for both $w$ and $b$ *before* updating either.

* **Correct:** Calculate `tmp_w` and `tmp_b` using the original values, then update.
* **Incorrect:** Updating $w$ first and then using the *new* $w$ to calculate the gradient for $b$. This "warps" the coordinate system mid-step.

### 4. Local vs. Global Minima

* **Convex (Bowl-shaped):** One single bottom point (**Global Minimum**). Found in Linear Regression.
* **Non-Convex:** Many "valleys" (**Local Minima**). The algorithm can get stuck in a sub-optimal spot.

---

**Key Insight:** As you approach the bottom, the gradient $\frac{\partial}{\partial w}$ naturally gets smaller, so your steps automatically shorten even if $\alpha$ stays the same.


### Why simultaneous updates matter in vector form

In NumPy, the safest pattern is:

```python
dw, db = gradients(X, y, w, b)  # computed from the same old parameters
w = w - alpha * dw
b = b - alpha * db
```

Both right-hand sides are evaluated before the names are rebound. This preserves the mathematical update rule.


In [3]:
def linear_gradients(X: np.ndarray, y: np.ndarray,
                     w: np.ndarray, b: float) -> tuple[np.ndarray, float]:
    """Compute batch gradients using all m examples."""
    m = len(y)
    errors = predict_linear(X, w, b) - y   # shape: (m,)
    dw = X.T @ errors / m                  # (n,m) @ (m,) -> (n,)
    db = float(errors.mean())              # scalar
    return dw, db


def fit_linear_gd(
    X: np.ndarray,
    y: np.ndarray,
    learning_rate: float = 0.05,
    steps: int = 500,
) -> tuple[np.ndarray, float, list[float]]:
    """Fit linear regression with batch gradient descent."""
    w = np.zeros(X.shape[1])
    b = 0.0
    history: list[float] = []

    for step in range(steps):
        dw, db = linear_gradients(X, y, w, b)
        w -= learning_rate * dw
        b -= learning_rate * db
        history.append(mse_cost(X, y, w, b))

    return w, b, history


w, b, history = fit_linear_gd(X, y)
print("Estimated w:", w)
print("Estimated b:", b)
print("Cost:", history[0], "->", history[-1])


Estimated w: [2.4981]
Estimated b: -1.007212101972397
Cost: 7.059285079783289 -> 0.32495037465426757


### Deriving the linear-regression gradient

For one example, define the prediction error:

$$
e^{(i)}=\hat y^{(i)}-y^{(i)}
=\mathbf w^T\mathbf x^{(i)}+b-y^{(i)}.
$$

Applying the chain rule:

$$
\frac{\partial}{\partial w_j}\frac12(e^{(i)})^2
=e^{(i)}\frac{\partial e^{(i)}}{\partial w_j}
=e^{(i)}x_j^{(i)}.
$$

Averaging across all examples gives:

$$
\frac{\partial J}{\partial w_j}
=\frac1m\sum_{i=1}^{m}e^{(i)}x_j^{(i)},\qquad
\frac{\partial J}{\partial b}
=\frac1m\sum_{i=1}^{m}e^{(i)}.
$$

Matrix notation collects all $n$ weight derivatives into one expression:

$$
\nabla_{\mathbf w}J=\frac1mX^T(X\mathbf w+b-\mathbf y).
$$


In [4]:
# Gradient checking: compare the analytical derivative with a
# finite-difference approximation.
test_w = np.array([0.3])
test_b = -0.2
analytical_dw, analytical_db = linear_gradients(X, y, test_w, test_b)

epsilon = 1e-5
numerical_dw = (
    mse_cost(X, y, test_w + epsilon, test_b)
    - mse_cost(X, y, test_w - epsilon, test_b)
) / (2 * epsilon)
numerical_db = (
    mse_cost(X, y, test_w, test_b + epsilon)
    - mse_cost(X, y, test_w, test_b - epsilon)
) / (2 * epsilon)

print("dw analytical / numerical:", analytical_dw[0], numerical_dw)
print("db analytical / numerical:", analytical_db, numerical_db)


dw analytical / numerical: -5.886718438613522 -5.8867184386635065
db analytical / numerical: 0.8241921070137573 0.8241921070517576


#### Reading the gradient as a sensitivity vector

With two features,

$$
\mathbf w=
\begin{bmatrix}w_1\\w_2\end{bmatrix},
\qquad
\nabla_{\mathbf w}J=
\begin{bmatrix}
\partial J/\partial w_1\\
\partial J/\partial w_2
\end{bmatrix}.
$$

Each component answers a separate “what if” question:

- $\partial J/\partial w_1$: how quickly would the cost change if only $w_1$ changed?
- $\partial J/\partial w_2$: how quickly would the cost change if only $w_2$ changed?

Together, they form a vector pointing toward the fastest local increase in cost. Gradient descent moves in the opposite direction.

For a small parameter change $\Delta\mathbf w$:

$$
\Delta J\approx
\nabla_{\mathbf w}J^T\Delta\mathbf w.
$$

This local linear approximation is the central connection between multivariable calculus and optimization.


In [5]:
# A two-feature example: every gradient component has the same
# position and meaning as the corresponding weight.
X_two = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
])
y_two = np.array([5.0, 4.0, 10.0])
w_two = np.array([1.0, 1.0])
b_two = 0.0

errors_two = X_two @ w_two + b_two - y_two
gradient_w = X_two.T @ errors_two / len(y_two)
gradient_b = errors_two.mean()

print("errors:", errors_two)
print("gradient_w:", gradient_w, "shape:", gradient_w.shape)
print("gradient_b:", gradient_b)


errors: [-2. -1. -3.]
gradient_w: [-4.3333 -5.6667] shape: (2,)
gradient_b: -2.0


## 3. Fixed Learning Rate, Batch Gradient Descent, and Convergence

A fixed learning rate does **not** imply fixed-size parameter updates. The update magnitude is `learning_rate × gradient`; near a smooth minimum, the gradient approaches zero, so the steps naturally shrink.

### Batch gradient descent

“Batch” means that every update uses all $m$ training examples:

$$
w_j := w_j-lpha
rac1m\sum_{i=1}^{m}
\left(f_{\mathbf w,b}(\mathbf x^{(i)})-y^{(i)}
ight)x_j^{(i)}.
$$

**Characteristics**

- The gradient is deterministic for fixed data and parameters.
- For a smooth convex linear-regression objective and an appropriate learning rate, the cost decreases toward the global minimum.
- Each update becomes expensive when $m$ is very large.
- Mini-batch methods trade some smoothness for cheaper, more frequent updates.

### Practical checks

1. Compute all gradients before updating any parameter.
2. Plot the cost against iterations.
3. If cost increases or oscillates, first test a much smaller learning rate.
4. Scale features when their ranges differ substantially.


In [6]:
# Visualize the fitted line, the convex cost surface, and convergence.
w_values = np.linspace(0, 5, 100)
b_values = np.linspace(-3, 1, 100)
cost_grid = np.empty((len(b_values), len(w_values)))

for row, b_candidate in enumerate(b_values):
    for col, w_candidate in enumerate(w_values):
        cost_grid[row, col] = mse_cost(
            X, y, np.array([w_candidate]), b_candidate
        )

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X[:, 0], y, alpha=0.45, label="data")
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
axes[0].plot(x_line[:, 0], predict_linear(x_line, w, b),
             color="crimson", linewidth=2, label="fitted model")
axes[0].set(title="Data and fitted model", xlabel="x", ylabel="y")
axes[0].legend()

contour = axes[1].contour(w_values, b_values, cost_grid, levels=18)
axes[1].clabel(contour, inline=True, fontsize=7)
axes[1].scatter(w[0], b, color="crimson", label="minimum")
axes[1].set(title="Convex cost contours", xlabel="w", ylabel="b")
axes[1].legend()

axes[2].plot(history)
axes[2].set(title="Learning curve", xlabel="iteration", ylabel="J(w,b)")
axes[2].set_yscale("log")

plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_62410/2173216914.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Multiple Linear Regression

### 1. Model Definition (Multiple Variables)
When we move from one feature to $n$ features, the model expands to account for each input variable $x_j$ with its own weight $w_j$.

For a model with $n=10$ features, the prediction function is:
$$f_{w,b}(x) = w_1 x_1 + w_2 x_2 + w_3 x_3 + \dots + w_{10} x_{10} + b$$

* **$x_j$**: The $j^{th}$ feature.
* **$w_j$**: The weight (parameter) for the $j^{th}$ feature, representing its relative importance to the prediction.
* **$b$**: The bias term (a single number).

---

### 2. Parameters and Vectors
To manage many features efficiently, we group the parameters and features into **vectors**. This is the foundation of modern machine learning implementation.

* **Weight Vector ($\mathbf{w}$):** $\mathbf{w} = [w_1, w_2, \dots, w_n]$
* **Feature Vector ($\mathbf{x}$):** $\mathbf{x} = [x_1, x_2, \dots, x_n]$
* **Bias ($b$):** A scalar value.



---

### 3. Vectorization & The Dot Product
Instead of writing a long summation or using a `for` loop in code (which is slow), we simplify the model using the **dot product**.

**Mathematical Simplification:**
$$f_{\mathbf{w},b}(\mathbf{x}) = \mathbf{w} \cdot \mathbf{x} + b$$

**Why use Dot Products?**
1.  **Code Clarity:** It turns dozens of lines of addition into a single operation.
2.  **Computational Efficiency:** Libraries like NumPy (Python) or Eigen (C++) use **parallel processing** in the CPU/GPU to calculate dot products much faster than a standard loop.

---

### 4. Gradient Descent for Multiple Variables
The update rule for Batch Gradient Descent remains conceptually the same, but it now applies to every parameter $w_j$ simultaneously.

For each $j = 1 \dots n$:
$$w_j := w_j - \alpha \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})x_j^{(i)}$$
$$b := b - \alpha \frac{1}{m} \sum_{i=1}^{m} (f_{\mathbf{w},b}(\mathbf{x}^{(i)}) - y^{(i)})$$

> **Note:** In Batch Gradient Descent, we calculate the error for the entire "batch" (all $m$ examples) before updating the vector $\mathbf{w}$.

---

### 5. Summary Table: Single vs. Multiple Variables

| Feature | Simple Linear Regression | Multiple Linear Regression |
| :--- | :--- | :--- |
| **Variables** | 1 ($x$) | $n$ ($x_1, x_2, \dots, x_n$) |
| **Parameters** | $w, b$ | $\mathbf{w}$ (vector), $b$ (scalar) |
| **Form** | $f = wx + b$ | $f = \mathbf{w} \cdot \mathbf{x} + b$ |


## 5. Vectorization in Machine Learning

### 1. The Core Concept
Vectorization is the process of replacing explicit `for` loops in your code with **array expressions**. In the context of Andrew Ng's course, this refers specifically to calculating the model's prediction and the gradient updates.

* **Non-Vectorized (Slow):** Loops through each feature $j$ one by one to calculate $w_j x_j$.
* **Vectorized (Fast):** Uses a single **dot product** operation to calculate the entire sum at once.

### 2. Implementation with NumPy
Using `np.dot()` allows the computer to utilize **Parallel Computing**. Instead of calculating $w_1x_1$, then $w_2x_2$ sequentially, a modern CPU/GPU can calculate multiple products simultaneously.



#### Code Reference
```python
import numpy as np

# Parameters: w is a vector (1-D array), b is a scalar (number)
w = np.array([2.0, 2.5, -3.3])
b = 4

# Input: x is a feature vector
x = np.array([10, 20, 30])

# Vectorized Calculation: f = w1x1 + w2x2 + w3x3 + b
f = np.dot(w, x) + b

print(f) # Result: -25.0
```

### 3. Why it Matters: The Hardware Level
The reason vectorization is a "must-have" rather than a "nice-to-have" in ML involves two hardware optimizations:
1.  **SIMD (Single Instruction, Multiple Data):** Modern processors have instructions that can perform the same operation (like multiplication) on a batch of data in a single clock cycle.
2.  **Parallelization:** In Batch Gradient Descent, when you have $m=100,000$ examples, vectorization allows the system to distribute the workload across multiple cores.

---

### 4. Mathematical Mapping
When reviewing your notes, remember that the dot product is mathematically equivalent to the summation of products:

$$\mathbf{w} \cdot \mathbf{x} = \sum_{j=1}^{n} w_j x_j = w_1x_1 + w_2x_2 + \dots + w_nx_n$$

> **Crucial Insight:** Vectorization isn't just about "shorter code." It is the difference between a model training in **10 seconds** versus **10 hours** when the dataset is large.


In [7]:
# One prediction: dot product between one feature vector and w.
x_one = np.array([10.0, 20.0, 30.0])   # shape: (n,)
w_one = np.array([2.0, 2.5, -3.3])     # shape: (n,)
b_one = 4.0
one_prediction = x_one @ w_one + b_one

# A batch of predictions: matrix-vector multiplication.
X_batch = np.array([
    [10.0, 20.0, 30.0],
    [12.0, 18.0, 25.0],
    [8.0, 24.0, 31.0],
])                                      # shape: (m,n)
batch_predictions = X_batch @ w_one + b_one

print("One prediction:", one_prediction)
print("Batch predictions:", batch_predictions)
print("Shape check:", X_batch.shape, "@", w_one.shape,
      "->", batch_predictions.shape)


One prediction: -25.0
Batch predictions: [-25.   -9.5 -22.3]
Shape check: (3, 3) @ (3,) -> (3,)


## 6. The Normal Equation: A Closed-Form Alternative

Linear regression can also be solved without iterative gradient descent. If the bias is included as a column of ones in the design matrix, the least-squares solution is:

$$
\boldsymbol\theta = X^{+}\mathbf y,
$$

where $X^{+}$ is the Moore–Penrose pseudoinverse. The familiar formula
$(X^TX)^{-1}X^T\mathbf y$ assumes invertibility; using a pseudoinverse or a least-squares solver is numerically safer.

**Advantages**

- no learning rate;
- no iteration or convergence monitoring;
- no feature scaling required for the optimization path.

**Limitations**

- applies to linear least-squares models, not arbitrary ML objectives;
- becomes expensive for very large feature counts;
- does not remove the need for careful data splitting and evaluation.


In [8]:
# Add a column of ones so theta contains [bias, weight].
X_with_bias = np.column_stack([np.ones(len(X)), X])

# lstsq solves the least-squares problem without explicitly inverting X.T @ X.
theta, residuals, rank, singular_values = np.linalg.lstsq(
    X_with_bias, y, rcond=None
)
print("Closed-form [b, w]:", theta)
print("Gradient-descent [b, w]:", np.r_[b, w])


Closed-form [b, w]: [-1.0072  2.4981]
Gradient-descent [b, w]: [-1.0072  2.4981]


## 7. Feature Scaling

### 1. The Relationship Between Range and Parameters
In a model where features have vastly different scales (e.g., $x_1$ = house size in sq ft [0–2000], $x_2$ = number of bedrooms [1–5]):
* **Large Range ($x_1$)**: Small changes in $w_1$ lead to huge changes in the prediction. Therefore, the model needs a **small parameter** $w_1$ to compensate.
* **Small Range ($x_2$)**: The model typically needs a **large parameter** $w_2$ to have any meaningful impact on the output.

### 2. Impact on Gradient Descent (The "Oval" Problem)
If features are not scaled, the contour plot of the cost function $J(w, b)$ becomes a **tall, skinny oval** (highly eccentric).

* **The Problem**: Gradient Descent always moves perpendicular to the contour lines. In an oval-shaped cost function, the gradient doesn't point directly toward the center (minimum).
* **The Result**: The optimization may **bounce back and forth** (oscillate) for a long time before reaching the center, making the process extremely slow.



---

### 3. Solution: Feature Rescaling
By rescaling, we transform the contours into **circles**. This allows Gradient Descent to point directly toward the global minimum, leading to a much faster and smoother "straight-line" convergence.

#### Methods to Rescale:
1.  **Divide by Maximum**: 
    $$x_{new} = \frac{x}{max}$$
    Simple, but sensitive to outliers.
2.  **Mean Normalization**: 
    $$x_{i} = \frac{x_i - \mu_i}{max - min}$$
    Centers the data around zero (range usually $-0.5$ to $0.5$).
3.  **Z-score Normalization (Standardization)**: 
    $$x_{i} = \frac{x_i - \mu_i}{\sigma_i}$$
    Uses the mean ($\mu$) and standard deviation ($\sigma$). This is the most robust method for many algorithms.

---

### 4. The Goal Range
The aim is to get each feature $x_i$ into a similar scale, typically:
$$-1 \le x_i \le 1$$
* **Acceptable ranges**: Roughly $-3$ to $+3$ or $-0.3$ to $+0.3$ are usually fine.
* **When to scale**: If a feature is too large (e.g., $-100$ to $100$) or too small (e.g., $-0.0001$ to $0.0001$), rescaling is mandatory for Gradient Descent to function properly.


### Important refinement: fit scaling statistics on training data only

The mean and standard deviation are learned parameters. If they are computed using validation or test rows, information from those rows leaks into training. A Scikit-learn `Pipeline` handles this correctly during cross-validation.

Scaling does not make every algorithm more accurate by itself. It mainly improves numerical conditioning and matters most for distance-based methods, gradient-based optimization, and regularized models.


In [9]:
X_scale_demo = np.array([
    [50_000.0, 20.0],
    [80_000.0, 35.0],
    [120_000.0, 50.0],
])

scaler = StandardScaler()
X_standardized = scaler.fit_transform(X_scale_demo)

print("Original feature ranges:")
print(np.ptp(X_scale_demo, axis=0))
print("Standardized data:")
print(X_standardized)
print("Means:", X_standardized.mean(axis=0))
print("Standard deviations:", X_standardized.std(axis=0))


Original feature ranges:
[70000.    30.]
Standardized data:
[[-1.1625 -1.2247]
 [-0.1162  0.    ]
 [ 1.2787  1.2247]]
Means: [0. 0.]
Standard deviations: [1. 1.]


### Curvature and conditioning: the calculus view

If the cost changes much faster in one parameter direction than another, its contour lines become elongated. The second-derivative matrix (the Hessian) has very different curvature values along different directions.

Gradient descent then takes a step that is safe for the steep direction but inefficient for the flat direction, producing a zigzag path. Scaling features often makes these curvatures more similar, improving the problem's **conditioning**.

You do not need to compute a Hessian to use this insight. The practical signs are:

- feature ranges differ by orders of magnitude;
- the learning curve decreases very slowly;
- parameters oscillate while moving toward the minimum.


## 8. Diagnosing and Optimizing Gradient Descent

To ensure Gradient Descent is effectively minimizing the cost function $J(\vec{w},b)$, you must monitor its progression. Relying blindly on the algorithm without diagnostics can lead to non-convergence.

**1. The Learning Curve**
*   **What it is:** A plot with the number of iterations on the x-axis and the value of the cost function $J(\vec{w},b)$ on the y-axis.
*   **The Essence:** It visually represents the optimization trajectory. A healthy learning curve should decrease strictly monotonically (going down at every single step) and eventually flatten out (plateau), indicating it has reached a minimum.


**2. Verifying Convergence**
*   **Automatic Convergence Test:** You can set an algorithm to stop when $J(\vec{w},b)$ decreases by less than a very small threshold, $\epsilon$ (e.g., $\epsilon = 10^{-3}$), in a single iteration.
*   **The Reality:** While $\epsilon$ is mathematically convenient, choosing the "right" $\epsilon$ is difficult. In practice, visual inspection of the learning curve is a far more reliable indicator of whether the algorithm has truly converged or is just moving slowly.

**3. Choosing the Learning Rate ($\alpha$)**
*   **The Mathematical Guarantee:** With a *small enough* learning rate, mathematical proofs guarantee that $J(\vec{w},b)$ will decrease on every single iteration. 
*   **The Trade-off:** 
    *   If $\alpha$ is too small: Gradient descent works, but it takes an impractically long time to converge.
    *   If $\alpha$ is too large: The algorithm overshoots the minimum. $J(\vec{w},b)$ may oscillate or even diverge (increase over time).
*   **Debugging Strategy:** If your learning curve goes up, or goes up and down irregularly, your $\alpha$ is almost certainly too large. Drop it significantly to verify the code works, then gradually increase it to find the most efficient rate.

---


In [10]:
# Compare several learning rates on the same linear-regression problem.
learning_rates = [0.005, 0.05, 0.25]

plt.figure(figsize=(8, 4))
for alpha in learning_rates:
    _, _, alpha_history = fit_linear_gd(
        X, y, learning_rate=alpha, steps=150
    )
    plt.plot(alpha_history, label=f"alpha={alpha}")

plt.yscale("log")
plt.xlabel("Iteration")
plt.ylabel("Cost J(w,b)")
plt.title("Learning-rate comparison")
plt.legend()
plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_62410/1525957731.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Feature Engineering and Polynomial Regression

**1. Feature Engineering**
*   **The Essence:** You are not restricted to the raw data you are given. Feature engineering is the process of using domain knowledge to design new features by transforming or combining existing ones. For example, if you have the `frontage` and `depth` of a plot of land, multiplying them creates a new feature: `area`. This allows a simple linear model to capture complex, interactive relationships.

**2. Polynomial Regression & The Necessity of Feature Scaling**
*   **The Concept:** You can fit non-linear curves to your data by taking standard features and raising them to powers (squaring, cubing, etc.). E.g., $f_{\vec{w},b}(x) = w_1x + w_2x^2 + w_3x^3 + b$.
*   **The Hidden Trap (Why Feature Scaling is Critical here):** If your base feature $x$ (e.g., house size) ranges from 1 to 1,000, then $x^2$ ranges from 1 to 1,000,000, and $x^3$ ranges from 1 to 1,000,000,000. 
*   **The Consequence:** The ranges of your features are now vastly different. The cost function's contour plot will become extremely elongated and narrow. Gradient descent will bounce inefficiently back and forth across this narrow valley. **You must apply Feature Scaling (like Z-score normalization) whenever you use Polynomial Regression.**

---


### Polynomial regression remains linear in its parameters

The model $w_1x+w_2x^2+w_3x^3+b$ is nonlinear in $x$, but it is linear in the parameters $w_1,w_2,w_3,b$. We create transformed features and then fit an ordinary linear model.

Degree controls flexibility:

- too low: high bias / underfitting;
- too high without enough regularization: high variance / overfitting;
- choose degree and regularization strength using cross-validation.


In [11]:
# Pipeline prevents leakage: polynomial expansion and scaling are refit
# independently inside every cross-validation fold.
polynomial_ridge = Pipeline([
    ("poly", PolynomialFeatures(include_bias=False)),
    ("scale", StandardScaler()),
    ("model", Ridge()),
])

search = GridSearchCV(
    polynomial_ridge,
    param_grid={
        "poly__degree": [1, 2, 3, 5],
        "model__alpha": [0.01, 0.1, 1.0, 10.0],
    },
    scoring="neg_root_mean_squared_error",
    cv=5,
)
search.fit(X, y)
print("Best parameters:", search.best_params_)
print("Cross-validated RMSE:", -search.best_score_)


Best parameters: {'model__alpha': 0.01, 'poly__degree': 1}
Cross-validated RMSE: 0.8097851884920167


# Part II — Classification and Logistic Regression

Regression predicts a continuous target. Classification predicts a discrete class, often by first estimating a probability and then applying a decision threshold.


## 10. The Transition to Classification

**1. Binary Classification**
*   **The Concept:** Predicting a discrete, finite set of outcomes. In binary classification, the outcome $y$ can only be 0 (Negative Class) or 1 (Positive Class). Example: 0 = Benign tumor, 1 = Malignant tumor.

**2. Why Linear Regression Fails for Classification**
*   **The Threshold Method:** You might attempt to use Linear Regression by setting a threshold: if the predicted value $f(x) \ge 0.5$, predict $y=1$; if $f(x) < 0.5$, predict $y=0$.
*   **The Outlier Problem:** Linear regression tries to fit a line to *all* data points minimizing the squared error. If you add a single extreme outlier (e.g., a massive, highly malignant tumor far to the right of the graph), the linear regression line gets violently pulled toward that outlier to minimize its large squared error.
*   **The Essence of the Failure:** Because the line tilts toward the outlier, the point where the line crosses the 0.5 threshold shifts dramatically. This shifted **Decision Boundary** will now incorrectly classify perfectly obvious positive cases as negative. Linear regression is too sensitive to extreme values to be reliable for classification.


---


## 11. Logistic Regression

To fix the outlier problem, we need an algorithm whose output is strictly bounded between 0 and 1, regardless of how extreme the input $x$ is.

**1. The Sigmoid (Logistic) Function**
*   **The Formula:** 
    $$g(z) = \frac{1}{1 + e^{-z}}$$
*   **The Shape:** It creates an S-shaped curve that asymptotes at 0 and 1. It crosses 0.5 exactly when $z = 0$.




**2. Building the Logistic Regression Model**
*   Instead of outputting the linear equation directly, Logistic Regression takes the linear equation and plugs it *inside* the Sigmoid function.
*   Step 1: Calculate the linear output: $z = \vec{w} \cdot \vec{x} + b$
*   Step 2: Apply the Sigmoid function: $f_{\vec{w},b}(\vec{x}) = g(z) = g(\vec{w} \cdot \vec{x} + b)$
*   **Interpretation:** The output $f_{\vec{w},b}(\vec{x})$ is interpreted as a **probability**. If the output is 0.7, the model is stating there is a 70% probability that $y=1$ given the input features $\vec{x}$.

---


In [12]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    """Numerically stable enough for this demonstration."""
    z = np.clip(z, -500, 500)
    return 1 / (1 + np.exp(-z))


z = np.linspace(-10, 10, 400)
probabilities = sigmoid(z)

plt.figure(figsize=(8, 4))
plt.plot(z, probabilities, linewidth=2.5,
         label=r"$\sigma(z)=1/(1+e^{-z})$")
plt.axhline(0.5, color="gray", linestyle="--")
plt.axvline(0, color="gray", linestyle="--")
plt.scatter([0], [0.5], color="crimson", zorder=3)
plt.annotate("threshold: z=0, p=0.5", xy=(0, 0.5), xytext=(1.5, 0.35),
             arrowprops={"arrowstyle": "->"})
plt.xlabel("Linear score z")
plt.ylabel("Probability")
plt.title("The sigmoid function")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_62410/3561291126.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. The Decision Boundary

**1. The Mathematical Definition**
The decision boundary is the exact line (or hyperplane) that separates the area where we predict $y=1$ from the area where we predict $y=0$.
*   We predict $y=1$ when $g(z) \ge 0.5$.
*   Looking at the Sigmoid function, $g(z) \ge 0.5$ happens strictly when $z \ge 0$.
*   Therefore, since $z = \vec{w} \cdot \vec{x} + b$, the decision boundary is exactly the mathematical set of points where:
    $$\vec{w} \cdot \vec{x} + b = 0$$

**2. Non-Linear Decision Boundaries**
*   **The Essence:** The decision boundary equation $\vec{w} \cdot \vec{x} + b = 0$ is just a line. How do we classify data that cannot be separated by a straight line?
*   **The Solution:** We combine Logistic Regression with **Polynomial Feature Engineering**.
*   By adding higher-order terms (e.g., $x_1^2$, $x_2^2$) into the $z$ equation, our condition becomes something like $w_1x_1^2 + w_2x_2^2 + b = 0$. 
*   This is the mathematical equation for complex geometric shapes. For example, $x_1^2 + x_2^2 - 1 = 0$ creates a circular decision boundary, allowing Logistic Regression to isolate clusters of data surrounded by other data.


In [13]:
# A circular decision boundary comes from polynomial features.
grid_x1 = np.linspace(-2, 2, 300)
grid_x2 = np.linspace(-2, 2, 300)
xx1, xx2 = np.meshgrid(grid_x1, grid_x2)

# z = x1^2 + x2^2 - 1; boundary is z=0, a unit circle.
z_grid = xx1 ** 2 + xx2 ** 2 - 1

plt.figure(figsize=(5, 5))
plt.contourf(xx1, xx2, z_grid >= 0, alpha=0.25, cmap="coolwarm")
plt.contour(xx1, xx2, z_grid, levels=[0], colors="black", linewidths=2)
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title(r"Nonlinear boundary: $x_1^2+x_2^2-1=0$")
plt.axis("equal")
plt.tight_layout()
plt.show()


/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_62410/1781724726.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


# Part III — Logistic Loss, Generalization, and Reliable Evaluation


## 13. Problem Setup & Notation
* **$m$**: Number of training examples in the dataset.
* **$n$**: Number of features per example.
* **$\vec{x}^{(i)}$**: The feature vector for the $i$-th training example.
* **$y^{(i)}$**: The true target label for the $i$-th example ($y \in \{0, 1\}$).
* **$\vec{w}, b$**: The parameters (weights and bias) of the model.


## 14. The Cost Function (Why Mean Squared Error Fails)
In Linear Regression, we use the Squared Error Cost Function. If we try to use Squared Error for Logistic Regression, plotting the cost against the parameters results in a **non-convex** curve (it looks "wavy" with many local minima). Gradient descent is not guaranteed to find the global minimum here.

### The Solution: Logistic Loss (Binary Cross-Entropy)
Instead, we use a loss function derived from **Maximum Likelihood Estimation (MLE)**. This guarantees a **convex** cost function (a single "bowl" shape), ensuring gradient descent will always converge to the global minimum.

Because $y$ can only be exactly 0 or exactly 1, we can condense the piecewise loss into a single elegant equation:
$$L(f_{\vec{w},b}(\vec{x}), y) = -y \log(f_{\vec{w},b}(\vec{x})) - (1 - y) \log(1 - f_{\vec{w},b}(\vec{x}))$$

The **Total Cost Function** $J(\vec{w}, b)$ is the average of the loss across all $m$ examples:
$$J(\vec{w}, b) = -\frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(f_{\vec{w},b}(\vec{x}^{(i)})) + (1 - y^{(i)}) \log(1 - f_{\vec{w},b}(\vec{x}^{(i)})) \right]$$


### Why log loss strongly penalizes confident mistakes

For a positive example ($y=1$), the loss is $-\log(p)$. Predicting $p=0.9$ gives a small loss, while predicting $p=0.001$ gives a very large loss. The same logic is mirrored for a negative example.

In code, probabilities are clipped away from exactly 0 and 1 before applying `log`, preventing `log(0)`.


In [14]:
def binary_log_loss(y_true: np.ndarray,
                    probabilities: np.ndarray) -> float:
    probabilities = np.clip(probabilities, 1e-12, 1 - 1e-12)
    losses = -(
        y_true * np.log(probabilities)
        + (1 - y_true) * np.log(1 - probabilities)
    )
    return float(losses.mean())


y_examples = np.array([1, 1, 0, 0])
good_predictions = np.array([0.9, 0.8, 0.2, 0.1])
bad_predictions = np.array([0.1, 0.2, 0.8, 0.9])

print("Loss for good predictions:", binary_log_loss(y_examples, good_predictions))
print("Loss for bad predictions: ", binary_log_loss(y_examples, bad_predictions))


Loss for good predictions: 0.16425203348601802
Loss for bad predictions:  1.956011502714073


## 15. Optimization: Gradient Descent
To find the optimal parameters $\vec{w}$ and $b$, we iteratively update them in the direction of the steepest descent (the negative gradient).

**The Update Rules:**
Repeat until convergence:
$$w_j = w_j - \alpha \frac{\partial}{\partial w_j} J(\vec{w}, b)$$
$$b = b - \alpha \frac{\partial}{\partial b} J(\vec{w}, b)$$
*(Where $\alpha$ is the learning rate).*

The chain-rule derivation is shown in the next section.

The final gradients are:
$$\frac{\partial}{\partial w_j} J(\vec{w}, b) = \frac{1}{m} \sum_{i=1}^m (f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)})x_j^{(i)}$$
$$\frac{\partial}{\partial b} J(\vec{w}, b) = \frac{1}{m} \sum_{i=1}^m (f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)})$$

**Crucial Insight:** *Why does this look exactly like the Linear Regression gradient?* The equations are structurally identical, but the **hypothesis** $f_{\vec{w},b}(\vec{x})$ is fundamentally different. In linear regression, $f(x) = \vec{w} \cdot \vec{x} + b$. In logistic regression, $f(x) = \sigma(\vec{w} \cdot \vec{x} + b)$. The non-linearity is hidden inside the $f(x)$ term.

### Implementation Enhancements
* **Vectorization:** Instead of using `for` loops across $m$ examples and $n$ features, represent $\vec{X}$ as a matrix and $\vec{w}$ as a vector. Matrix multiplication (using libraries like NumPy) computes predictions and gradients orders of magnitude faster.
* **Feature Scaling (Z-score normalization):** If features are on vastly different scales, the convex "bowl" of the cost function becomes skewed and narrow. Gradient descent will oscillate wildly. Scaling features ensures the contours are circular, allowing for a much faster, direct path to the minimum.


### Chain-rule derivation for logistic regression

For one example:

$$
z=\mathbf w^T\mathbf x+b,\qquad
p=\sigma(z),\qquad
L=-y\log p-(1-y)\log(1-p).
$$

The necessary derivatives are:

$$
\frac{\partial L}{\partial p}
=-\frac{y}{p}+\frac{1-y}{1-p},
\qquad
\frac{\partial p}{\partial z}=p(1-p).
$$

Multiplying and simplifying:

$$
\frac{\partial L}{\partial z}
=\left(-\frac{y}{p}+\frac{1-y}{1-p}\right)p(1-p)
=p-y.
$$

Since $\partial z/\partial w_j=x_j$:

$$
\frac{\partial L}{\partial w_j}=(p-y)x_j,
\qquad
\frac{\partial L}{\partial b}=p-y.
$$

This cancellation explains why the final gradient has the same **shape** as the linear-regression gradient even though the model and loss are different.


## 16. Classification Metrics and Thresholds

Accuracy is useful only when its assumptions match the problem. For imbalanced classes or unequal error costs, inspect:

- **Precision:** among predicted positives, how many are truly positive?
- **Recall:** among actual positives, how many did the model find?
- **F1:** harmonic mean of precision and recall.
- **Confusion matrix:** counts of true/false positives/negatives.
- **Decision threshold:** converts probability into a class and controls the precision–recall tradeoff.


In [15]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data,
    cancer.target,
    test_size=0.2,
    random_state=42,
    stratify=cancer.target,  # preserve the class ratio
)

classifier = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=2_000)),
])
classifier.fit(X_train, y_train)
predicted = classifier.predict(X_test)

print("Accuracy:", accuracy_score(y_test, predicted))
print(classification_report(
    y_test, predicted, target_names=cancer.target_names
))

ConfusionMatrixDisplay.from_estimator(
    classifier,
    X_test,
    y_test,
    display_labels=cancer.target_names,
    cmap="Blues",
)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()


Accuracy: 0.9824561403508771
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



/var/folders/fs/nqhyfr317t17_3shs0v7ygdr0000gn/T/ipykernel_62410/2071497747.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [16]:
# Lower thresholds usually increase recall and reduce precision.
positive_probability = classifier.predict_proba(X_test)[:, 1]

for threshold in [0.3, 0.5, 0.7]:
    threshold_prediction = (positive_probability >= threshold).astype(int)
    print(f"\nThreshold = {threshold}")
    print(classification_report(
        y_test,
        threshold_prediction,
        target_names=cancer.target_names,
        zero_division=0,
    ))



Threshold = 0.3
              precision    recall  f1-score   support

   malignant       1.00      0.95      0.98        42
      benign       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Threshold = 0.5
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114


Threshold = 0.7
              precision    recall  f1-score   support

   malignant       0.89      0.98      0.93        42
      benign       0.99      0.93      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.95      0.94       114
weighted avg       0.95

## 17. The Bias-Variance Tradeoff
When training models, we are trying to balance two opposing sources of error:

1.  **Underfitting (High Bias):** The model is too simple. It makes strong, incorrect assumptions (bias) about the data and fails to capture the underlying pattern. It performs poorly on both the training and test sets.
2.  **Generalization (Just Right):** The model captures the true pattern and ignores the noise.
3.  **Overfitting (High Variance):** The model is excessively complex. It fits the training set perfectly—memorizing the exact data points and their random noise. However, slight changes in the data (variance) cause wild changes in the prediction. It fails completely on new, unseen data.


## 18. Addressing Overfitting: Regularization
If your model is overfitting, you have two primary options:
1.  **Feature Selection:** Manually or algorithmically discard features that do not carry enough useful information.
2.  **Regularization:** Keep all the features, but constrain the magnitude of the parameters $w_j$. 

**Intuition:** Large parameter values allow the decision boundary to create sharp, complex curves to encapsulate every single outlier. By forcing the weights to be small, we "smooth out" the decision boundary, making the model more robust to noise.

### The Regularized Cost Function
We add a penalty term to the end of the cost function. $\lambda$ (lambda) is the regularization parameter that balances the goal of fitting the data against the goal of keeping parameters small.
$$J(\vec{w}, b) = \text{Original Cost} + \frac{\lambda}{2m} \sum_{j=1}^{n} w_j^2$$
*(Note: By convention, we do not regularize the bias term $b$, as it only shifts the function left/right and does not contribute to the "wiggliness" of the curve).*

### The Regularized Gradient Descent
Because the cost function changed, the derivative changes slightly. We must add the derivative of the penalty term to the weight update:
$$w_j = w_j - \alpha \left[ \left( \frac{1}{m} \sum_{i=1}^m (f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)})x_j^{(i)} \right) + \frac{\lambda}{m} w_j \right]$$

We can algebraically rearrange this to reveal how regularization actually works step-by-step:
$$w_j = w_j \left( 1 - \alpha \frac{\lambda}{m} \right) - \alpha (\text{Original Gradient})$$
Because $(1 - \alpha \frac{\lambda}{m})$ is a number just slightly less than 1, the algorithm literally shrinks the weight by a small percentage on every single iteration before applying the gradient update. This is why this form of regularization is often called **"Weight Decay."**

***


### Interpreting regularization hyperparameters

Libraries use different parameterizations:

- `Ridge(alpha=...)`: larger `alpha` means stronger regularization.
- `LogisticRegression(C=...)`: `C` is approximately inverse strength, so smaller `C` means stronger regularization.

Always tune the library's actual parameter using cross-validation rather than assuming one default is universally appropriate.


## 19. Regression Metrics

- **MAE:** average absolute error; robust and easy to interpret in the target's unit.
- **RMSE:** square root of mean squared error; gives large errors more influence.
- **R²:** improvement over predicting the training mean. It can be negative and is not a percentage accuracy.


In [17]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y, test_size=0.2, random_state=42
)

regression_pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0)),
])
regression_pipeline.fit(X_train_reg, y_train_reg)
regression_predictions = regression_pipeline.predict(X_test_reg)

print("MAE: ", mean_absolute_error(y_test_reg, regression_predictions))
print("RMSE:", mean_squared_error(y_test_reg, regression_predictions) ** 0.5)
print("R²:  ", r2_score(y_test_reg, regression_predictions))


MAE:  0.6764891934616967
RMSE: 0.8316902831275136
R²:   0.9646986019571597


## 20. Data Leakage

Leakage occurs when training uses information that would not be available at prediction time. It creates impressive offline scores that do not survive deployment.

**Common patterns**

1. Computing scaling or imputation statistics on the full dataset before splitting.
2. Including features created after the prediction event.
3. Placing near-duplicate records from the same user or device in both train and test sets.
4. Repeatedly inspecting the test set while tuning.

**Defenses**

- split before fitting preprocessing;
- use a `Pipeline`;
- split time-dependent data chronologically;
- use grouped splitting for grouped observations;
- reserve the test set for one final evaluation.


## 21. Reusable Modeling Checklist

1. Define what one row represents, the target, and the prediction time.
2. Choose a metric that matches the cost of errors.
3. Build a simple baseline.
4. Split data before any preprocessing that learns parameters.
5. Put preprocessing and the model in one Pipeline.
6. Use cross-validation for hyperparameter selection.
7. Compare training and validation performance.
8. Inspect individual errors, not only aggregate scores.
9. Evaluate the test set once and record data/version/seed/parameters.
10. After deployment, monitor drift, performance, and fairness.

## Exercises

1. Add a second feature to the hand-written linear regression and verify all shapes.
2. Make gradient descent diverge with a large learning rate, then recover it.
3. Compare the normal-equation solution with gradient descent on scaled and unscaled data.
4. Add regression outliers and compare the change in MAE and RMSE.
5. Tune polynomial degree and Ridge `alpha`; explain the bias–variance tradeoff.
6. Change the classification threshold and explain the precision–recall tradeoff.
7. Tune logistic regression `C`; explain why smaller `C` means stronger regularization.
8. Construct one leaky feature, observe the score, then remove it.
